# v6 FULL GAMES — tournament phase (Dell PowerEdge R7615)

EMCTS plays complete games against every experimental configuration, at every ratio and every
budget. **The only thing measured is who won** — search telemetry is the isolation phase's job
(`run_v6_jupyter.ipynb`), and this phase answers the question that one deliberately cannot.

Run the cells in order. **§4 (game-length probe) decides the turn cap and §5 decides the thread
count; both come before §6.** §6 is also the RESUME cell — re-run it verbatim after any stop.

Design: `analysis/EXPERIMENT_V6_FULL_GAMES_PLAN.md`. Machine placement: `analysis/V6_SERVER_RUNBOOK.md`.

| | |
|---|---|
| matchups | EMCTS vs 11 experimental configs **+ EMCTS vs EMCTS (control)** |
| budgets | 40,000 / 80,000 / 400,000 compute-units per turn |
| games | 200 per matchup → **7,200 total** |
| maps | 20 fixed, each played 10 times: **5 per orientation** |
| expected cost | ~900–1,250 CPU-hours; the 400k level is ~90% of it |


In [ ]:
# EDIT to the absolute path of the Tribes-master directory on the server:
TRIBES = '/root/Bachelor_Thesis/Tribes-master'
%cd $TRIBES


## 0. Environment check

Same checks as the isolation phase, plus one specific to this run: **the two phases must not run at
the same time.** They are separate output trees with separate locks, so nothing would be corrupted,
but they would compete for the same cores and every throughput number either one measures would be
meaningless.


In [ ]:
!java -version 2>&1 | head -3
print('--- CPU topology ---')
!lscpu | egrep 'Model name|^CPU\(s\)|Thread\(s\) per core|Core\(s\) per socket|Socket\(s\)|NUMA node\(s\)|MHz'
print('--- memory / disk ---')
!free -g | head -2
!df -h . | tail -1
print('--- other java processes (the v6 isolation run must NOT be active) ---')
!pgrep -af java || echo '(none - good)'
print('--- run locks currently held ---')
!find test_results -name '_running.lock' -exec sh -c 'echo "$1: $(cat $1)"' _ {} \; 2>/dev/null || true


## 1. Build

Recompiles everything from `src/`. `--release 11` matches the server's JDK — keep it, or the classes
will not load there.

This build carries one change to the shared engine: `Game.printGameResults()` is now gated on
`Constants.VERBOSE`. Interactive play is unchanged (it defaults to true); the tournament sets it
false so 7,200 games across ~90 threads do not interleave five lines each into the run log.


In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac --release 11 -nowarn -cp "lib/*" -d out @_srcs.txt 2>&1 | tail -5
!echo "classes: $(find out -name '*.class' | wc -l)"


## 2. Smoke test (~1 min) — correctness gate

32 checks. The ones that matter most:

- **§2** the orientation swap is a real exchange — identical board, capitals swapped, seats swapped —
  asserted on all 20 maps;
- **§1c** capital positions and distances actually vary across the map set (this catches the
  `java.util.Random` cross-seed correlation that made every map geometrically identical in the first
  version);
- **§5h** the EMCTS-vs-EMCTS control can report losses (both sides share a name, so a name-based
  winner would silently make the control a 100% win rate);
- **§6** the same games replay byte-identically across a thread pool;
- **§8** the runner restores the engine constants — a preceding thread calibration lowers
  `MAX_TURNS_CAPITALS` to its own 2–12 tick cap, and a leak there would play the whole
  tournament to the wrong cap with nothing in the output to say so.

A failure here means do not run the experiment.


In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.NewExperimentSmokeTestV6Full


## 3. The map set (instant)

Generates the 20 maps and writes `test_results/v6_full/_maps/maps.csv` with capital positions,
capital distance and per-start terrain and resource counts.

**Asymmetry is measured, not filtered.** Every map is played 5 times in each orientation, so a map
bias of any size cancels exactly within that map; filtering would narrow the map distribution the
result generalises over while removing a bias the schedule already removes. The numbers are here so
the write-up can state what the maps looked like.


In [ ]:
!java -Xmx4g -cp "out:lib/*" core.data.exp.FullGameMapSetV6 test_results/v6_full/_maps


In [ ]:
import pandas as pd
m = pd.read_csv('test_results/v6_full/_maps/maps.csv')
print(f'{len(m)} maps, board {m.BoardSize.iloc[0]}x{m.BoardSize.iloc[0]}')
print('capital distance :', dict(m.CapitalDistance.value_counts().sort_index()))
print('start asymmetry  : min %d  median %.1f  max %d' % (
    m.StartAsymmetry.min(), m.StartAsymmetry.median(), m.StartAsymmetry.max()))
print('distinct capital-A tiles:', m.groupby(['CapAX','CapAY']).ngroups,
      ' capital-B tiles:', m.groupby(['CapBX','CapBY']).ngroups)
print()
print('If either count is 1, the level seeds are correlated and these are not really 20 maps.')
m[['MapIndex','LevelSeed','CapAX','CapAY','SeatA','CapBX','CapBY','SeatB',
   'CapitalDistance','StartAsymmetry']]


## 4. Game-length probe — sets the turn cap

**Run this and read it before §6.**

A CAPITALS game ends either by conquest (one tribe holds every capital) or by the turn cap, after
which the score ranking decides. Those measure different things: conquest is playing strength, and
score at a fixed horizon is dominated in Tribes by technology count. The old Phase 2 tournament
capped at 50 and only 1.3% of games reached it — but those agents planned on a 500–1000 ms clock
and these are far stronger, so the cap has to be re-measured rather than inherited.

The probe plays real games under a loose safety cap and recommends the smallest cap that would
decide at most 2% of games. Games that reach the safety cap are **censored, not dropped**.

Cost: minutes at 40k/80k, up to ~1–2 h at 400k. Args: `[threads|auto] [safetyCap] [gamesPerLevel]`.


In [ ]:
!nohup java -Xmx200g -XX:+UseG1GC -cp "out:lib/*" \
    core.data.exp.GameLengthProbeV6 auto 150 60,60,24 > v6full_probe.log 2>&1 &
print('started; re-run the next cell to watch it')


In [ ]:
!tail -40 v6full_probe.log


### 4b. Set the turn cap

Read the RECOMMENDATION block above. If it differs from the compiled `RunV6FullGames.TURN_LIMIT`,
edit that constant, re-run §1, and only then continue.

It is a compile-time constant on purpose: the tournament's rules should be fixed and visible in the
source, and a dataset must never silently contain games played under two different caps.


In [ ]:
import re, pathlib
src = pathlib.Path('src/core/data/exp/RunV6FullGames.java').read_text()
print('compiled TURN_LIMIT =', re.search(r'TURN_LIMIT\s*=\s*(\d+)', src).group(1))
!grep -A2 'RECOMMENDED TURN CAP' v6full_probe.log || echo '(probe has not finished)'


## 5. Thread calibration — per budget level

A separate sweep from the isolation phase's, because the work item is different: a **game**, not a
turn. It sweeps thread counts on real (short) games and picks the smallest count within 3% of the
best *sustained* throughput — the second of two timed segments, so a point that only reaches peak on
boost clocks shows up as `ThrottlePct` rather than being chosen.

Its honest limitation: the games it times are short and therefore early-game, so its per-task heap
estimate is low. It applies a 2× derate before gating, and the live `LoadGovernor` during §6 is the
real protection.

Skippable — §6 calibrates any level with no `_tuned_threads_full_<cu>cu.txt` — but running it here
keeps the sweep out of the run's own log.


In [ ]:
!java -Xmx700g -XX:+UseG1GC -cp "out:lib/*" core.data.exp.ThreadCalibrationV6Full \
    test_results/v6_full/_calibration


In [ ]:
import pandas as pd, os
p = 'test_results/v6_full/_calibration/thread_calibration_full.csv'
if os.path.exists(p):
    c = pd.read_csv(p)
    display(c[['ComputeUnits','Threads','SustainedGamesPerMin','ThrottlePct','Speedup',
               'EfficiencyPct','ClockDropPct','PeakHeapMB','Chosen']])
    print()
    print('Chosen per level:')
    print(c[c.Chosen=='yes'][['ComputeUnits','Threads','SustainedGamesPerMin']].to_string(index=False))
else:
    print('not calibrated yet')


## 6. Run — and the RESUME cell

Launches detached so it survives a disconnect. **Re-run this same cell verbatim to resume:** finished
matchups are skipped whole, a partial matchup continues per game, and a directory lock refuses a
second writer over the same tree.

The EMCTS-vs-EMCTS control runs **first** at each level — cheapest matchup, most informative failure,
so it completes while there is still time to react.

Every game of a level goes into one pool rather than a pool per matchup, so the workers stay busy and
no matchup's tail waits on its single longest game.


In [ ]:
!nohup java -Xmx750g -XX:+UseG1GC -XX:MaxGCPauseMillis=500 \
    -cp "out:lib/*" core.data.exp.RunV6FullGames auto > v6full_run.log 2>&1 &
print('started - watch with the progress cell below')


## 7. Progress (re-run to refresh)

Read-only view over the output tree. Touches nothing the running job owns.


In [ ]:
import os, glob, pandas as pd
BASE = 'test_results/v6_full'
EXPECTED_PER_LEAF = 200
EXPECTED_LEAVES = 36

rows = []
for p in sorted(glob.glob(f'{BASE}/**/games.csv', recursive=True)):
    leaf = os.path.relpath(os.path.dirname(p), BASE)
    try:
        d = pd.read_csv(p)
    except Exception:
        continue
    rows.append(dict(leaf=leaf, games=len(d), pct=round(100*len(d)/EXPECTED_PER_LEAF),
                     emcts_win=round(100*d.EmctsWon.mean(),1) if len(d) else None,
                     conquest=round(100*(d.DecidedBy=='CAPITAL').mean()) if len(d) else None))
prog = pd.DataFrame(rows)
if len(prog):
    total, expected = prog.games.sum(), EXPECTED_LEAVES*EXPECTED_PER_LEAF
    print(f'{total:,} / {expected:,} games  ({100*total/expected:.1f}%)')
    display(prog)
else:
    print('no games yet')

print('--- checkpointed matchups ---')
!wc -l < test_results/v6_full/_completed.txt 2>/dev/null || echo 0
print('--- live process ---')
!pgrep -af RunV6FullGames || echo '(not running)'
!ps -o rss=,nlwp= -p $(pgrep -f RunV6FullGames | head -1) 2>/dev/null | awk '{printf "RSS %.1f GB, %d threads\n", $1/1048576, $2}'
print('--- log tail ---')
!tail -12 v6full_run.log


### 7b. Stopping

**SIGTERM, never `kill -9`.** SIGTERM is a cooperative stop: queued games are abandoned, running games
finish and write their row, and no matchup is marked complete unless it really is. A hard kill can
leave a torn final row — recoverable by §8, but there is no reason to create the work.


In [ ]:
!pkill -TERM -f RunV6FullGames && echo 'SIGTERM sent - watch v6full_run.log for the drain' || echo 'not running'


## 8. Output check — run before ANY analysis

Looks for what an interrupted or doubled run leaves (torn rows, duplicate GameIds) **and** for
schedule violations, which matter more: a duplicated row is obvious in a count, whereas a matchup
that played one orientation more often than the other looks like ordinary data and would bias the
win rate by exactly the amount the swap exists to remove.

It also prints the control arm's win rate, the conquest share and the cap-bind rate.


In [ ]:
!java -cp "out:lib/*" core.data.exp.VerifyV6FullOutput test_results/v6_full


In [ ]:
# Only if the cell above reported fixable problems (duplicates / torn rows):
# !java -cp "out:lib/*" core.data.exp.VerifyV6FullOutput test_results/v6_full --repair


## 9. Analyse

Win rates with **map-clustered** intervals (10 games share a map, so a plain binomial interval
understates uncertainty), the pooled per-family headline test, the per-ratio curve, and the
mirror-pair sign test.

**Read the control arm first.** If EMCTS-vs-EMCTS is not 50% at a level, that level's other numbers
are not interpretable.


In [ ]:
!python3 analysis/analyze_v6_full_games.py test_results/v6_full --outdir analysis/v6_full_games


In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('analysis/v6_full_games/RESULTS_V6_FULL_GAMES.md').read()))


In [ ]:
from IPython.display import Image, display
import os
for f in ['F1_winrate_by_ratio.png', 'F2_forest.png']:
    p = f'analysis/v6_full_games/figures/{f}'
    if os.path.exists(p):
        display(Image(p))
